In [1]:
import os
import io 
from typing import Generator, List
import gradio as gr 
from openai import OpenAI 
from dotenv import load_dotenv

c:\Users\omega.sithebe\OneDrive - Sabio Ltd\Documents\Projects\mcp_gradio_tutor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL_NAME = "gpt-4o-mini"

In [6]:
EXPLANATION_LEVELS = {
    1: "like I'm 5 years old",
    2: "like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field"
}

In [9]:
def explain_concept(question: str, level: int):
    """
    Stream an explanation of a concept based on
    the student's learning level.
    """

    # Check if the user entered a question
    if not question.strip():
        yield "Error: Question cannot be blank."
        return

    # Convert number into description
    level_desc = EXPLANATION_LEVELS.get(
        level,
        "clearly and concisely"
    )

    # System Prompt
    system_prompt = (
        f"You are a helpful AI Tutor. "
        f"Explain the following concept {level_desc}."
    )

    # Call OpenAI
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": question,
            },
        ],
        stream=True,
        temperature=0.7,
    )

    partial = ""

    for chunk in stream:

        delta = getattr(
            chunk.choices[0].delta,
            "content",
            None,
        )

        if delta:

            partial += delta

            yield partial


In [10]:
def summarize_text(text: str, compression_ratio: float = 0.3):
    """
    Stream a concise summary of the input text.
    """

    if not text.strip():
        yield "Error: Text cannot be blank."
        return

    ratio = max(0.1, min(compression_ratio, 0.8))

    system_prompt = (
        "You are a world-class summarizer. "
        f"Reduce the following text to about "
        f"{int(ratio * 100)}% of its original length."
    )

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": text,
            },
        ],
        stream=True,
        temperature=0.5,
    )

    partial = ""

    for chunk in stream:

        delta = getattr(
            chunk.choices[0].delta,
            "content",
            None,
        )

        if delta:

            partial += delta

            yield partial

In [11]:
def generate_flashcards(
    topic: str,
    num_cards: int = 5,
):
    """
    Generate study flashcards.
    """

    if not topic.strip():
        yield "Topic cannot be blank."
        return

    if num_cards < 1 or num_cards > 20:
        yield "Number of cards must be between 1 and 20."
        return

    system_prompt = (
        "You are an AI that creates study flashcards. "
        'Return each flashcard as JSON: '
        '{"q":"Question","a":"Answer"}'
    )

    user_prompt = (
        f"Create {num_cards} flashcards "
        f"about {topic}."
    )

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        stream=True,
        temperature=0.8,
    )

    partial = ""

    for chunk in stream:

        delta = getattr(
            chunk.choices[0].delta,
            "content",
            None,
        )

        if delta:

            partial += delta

            yield partial

In [12]:
def quiz_me(
    topic: str,
    level: int = 3,
    num_questions: int = 5,
):
    """
    Generate a multiple-choice quiz.
    """

    if not topic.strip():
        yield "Topic cannot be blank."
        return

    if num_questions < 1 or num_questions > 15:
        yield "Questions must be between 1 and 15."
        return

    level_desc = EXPLANATION_LEVELS.get(
        level,
        "Intermediate"
    )

    system_prompt = (
        "You are an AI quiz master. "
        f"Create {num_questions} multiple-choice questions "
        f"about {topic} {level_desc}. "
        "After all questions, include an ANSWER KEY."
    )

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            }
        ],
        stream=True,
        temperature=0.7,
    )

    partial = ""

    for chunk in stream:

        delta = getattr(
            chunk.choices[0].delta,
            "content",
            None,
        )

        if delta:

            partial += delta

            yield partial

In [13]:
def build_demo():

    with gr.Blocks() as demo:

        gr.Markdown("# AI Tutor MCP Toolkit – Demo Console")

        with gr.Tab("Explain Concept"):

            q = gr.Textbox(label="Concept / Question")

            lvl = gr.Slider(
                1,
                5,
                value=3,
                step=1,
                label="Explanation Level"
            )

            out1 = gr.Markdown()

            gr.Button("Explain").click(
                explain_concept,
                inputs=[q, lvl],
                outputs=out1,
            )

        with gr.Tab("Summarize Text"):

            txt = gr.Textbox(
                lines=8,
                label="Long Text"
            )

            ratio = gr.Slider(
                0.1,
                0.8,
                value=0.3,
                step=0.05,
                label="Compression Ratio"
            )

            out2 = gr.Markdown()

            gr.Button("Summarize").click(
                summarize_text,
                inputs=[txt, ratio],
                outputs=out2,
            )

        with gr.Tab("Flashcards"):

            topic_fc = gr.Textbox(label="Topic")

            n_fc = gr.Slider(
                1,
                20,
                value=5,
                step=1,
                label="# Cards"
            )

            out3 = gr.Markdown()

            gr.Button("Generate").click(
                generate_flashcards,
                inputs=[topic_fc, n_fc],
                outputs=out3,
            )

        with gr.Tab("Quiz Me"):

            topic_q = gr.Textbox(label="Topic")

            lvl_q = gr.Slider(
                1,
                5,
                value=3,
                step=1,
                label="Difficulty Level"
            )

            n_q = gr.Slider(
                1,
                15,
                value=5,
                step=1,
                label="# Questions"
            )

            out4 = gr.Markdown()

            gr.Button("Start Quiz").click(
                quiz_me,
                inputs=[topic_q, lvl_q, n_q],
                outputs=out4,
            )

    return demo

In [14]:
if __name__ == "__main__":

    print("Starting AI Tutor MCP Toolkit on port 7860...")

    build_demo().launch(
        server_name="0.0.0.0",
        server_port=7860,
        mcp_server=True,
    )

Starting AI Tutor MCP Toolkit on port 7860...
Error launching MCP server: The `mcp` package is required to use the Gradio MCP integration. Please install it with the `mcp` extra: `pip install gradio[mcp]`.
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\omega.sithebe\OneDrive - Sabio Ltd\Documents\Projects\mcp_gradio_tutor\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\omega.sithebe\OneDrive - Sabio Ltd\Documents\Projects\mcp_gradio_tutor\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\omega.sithebe\OneDrive - Sabio Ltd\Documents\Projects\mcp_gradio_tutor\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\omega.sithebe\OneDrive - Sabio Ltd\Documents\Projects\mcp_gradio_tutor